<a href="https://colab.research.google.com/github/takuya0724/sticker-pro/blob/main/StickerPro_v1_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install opencv-python pillow

In [2]:
from PIL import Image
from google.colab import files
import cv2
import numpy as np
import os
import zipfile
import shutil

OUTPUT_SIZE = 512
PADDING = 20
OUTPUT_FOLDER = "stickers"

In [3]:
print("4×4画像を選択してください")

uploaded = files.upload()
filename = list(uploaded.keys())[0]

img = Image.open(filename).convert("RGBA")
img = np.array(img)

h, w = img.shape[:2]

ROWS = 4
COLS = 4

cell_h = h // ROWS
cell_w = w // COLS

tiles = []

for y in range(ROWS):
    for x in range(COLS):

        crop = img[
            y*cell_h:(y+1)*cell_h,
            x*cell_w:(x+1)*cell_w
        ]

        tiles.append(crop)

print(f"分割完了：{len(tiles)}枚")

4×4画像を選択してください


Saving file_000000003774820698ff3ee84af91b31.png to file_000000003774820698ff3ee84af91b31 (3).png
分割完了：16枚


In [4]:
# ============================================================
# 背景透過処理
# ============================================================

transparent_tiles = []

for tile in tiles:

    img = tile.copy()

    # RGB取得
    rgb = img[:, :, :3]

    # 四隅の色を背景色と判断
    corners = np.array([
        rgb[0,0],
        rgb[0,-1],
        rgb[-1,0],
        rgb[-1,-1]
    ])

    bg_color = np.mean(corners, axis=0)

    # 背景との差
    diff = np.linalg.norm(
        rgb.astype(float) - bg_color,
        axis=2
    )

    # 透明化する範囲
    mask = diff < 35

    # アルファ変更
    img[:,:,3][mask] = 0

    transparent_tiles.append(img)

print("背景透過完了")

背景透過完了


In [5]:
# ============================================================
# 透過後の画像で余白カット
# ============================================================

trimmed_tiles = []

for tile in transparent_tiles:

    alpha = tile[:,:,3]

    coords = cv2.findNonZero(alpha)

    if coords is None:
        trimmed_tiles.append(tile)
        continue

    x, y, w, h = cv2.boundingRect(coords)

    margin = 10

    x1 = max(0, x - margin)
    y1 = max(0, y - margin)

    x2 = min(tile.shape[1], x + w + margin)
    y2 = min(tile.shape[0], y + h + margin)

    crop = tile[y1:y2, x1:x2]

    trimmed_tiles.append(crop)

print("透過後切り抜き完了")

透過後切り抜き完了


In [6]:
# ============================================================
# 512×512キャンバスへ配置
# ============================================================

canvas_size = 512

output_tiles = []

for i, tile in enumerate(trimmed_tiles):

    # 白紙キャンバス
    canvas = np.zeros(
        (canvas_size, canvas_size, 4),
        dtype=np.uint8
    )

    # タイルサイズ取得
    h, w = tile.shape[:2]

    # 512以内に収まるよう縮小
    scale = min(
        canvas_size / w,
        canvas_size / h,
        1
    )

    new_w = int(w * scale)
    new_h = int(h * scale)

    resized = cv2.resize(
        tile,
        (new_w, new_h),
        interpolation=cv2.INTER_AREA
    )

    # 中央配置
    x = (canvas_size - new_w) // 2
    y = (canvas_size - new_h) // 2

    canvas[y:y+new_h, x:x+new_w] = resized

    output_tiles.append(canvas)

print(f"512×512配置完了：{len(output_tiles)}枚")

512×512配置完了：16枚


In [7]:
# ============================================================
# 個別PNG出力（色修正版）
# ============================================================

from google.colab import files
import os
from PIL import Image
import shutil

output_dir = "line_stamp_png"

os.makedirs(output_dir, exist_ok=True)

for i, tile in enumerate(output_tiles, start=1):

    filename = f"{output_dir}/stamp_{i:02d}.png"

    img = Image.fromarray(tile, "RGBA")

    img.save(
        filename,
        "PNG"
    )

print("PNG出力完了")

# ZIP化
zip_name = "line_stamp_png.zip"

shutil.make_archive(
    "line_stamp_png",
    "zip",
    output_dir
)

print("ZIP作成完了")

files.download(zip_name)

/tmp/ipykernel_9966/1014837991.py:18: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(tile, "RGBA")


PNG出力完了
ZIP作成完了


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
tile = transparent_tiles[0]

print(
    "透明ピクセル:",
    (tile[:,:,3] == 0).sum()
)

透明ピクセル: 49280
